<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Answer:

I am continuing with Lane 4: CTR / Engagement Opportunity Scoring. I would frame this mainly as a ranking/scoring task. The system would give each page an opportunity score and rank the pages so that an SEO or content editor can review the most promising opportunities first.

Ranking is more useful here than simply classifying pages as good or bad because the main decision is which pages should be reviewed first.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Answer:

The starter dataset has no column that says whether a page was truly worth reviewing, so I will use a proxy (a measurable stand-in).

My proxy is missed clicks: how many more clicks a page would have received if it had the typical CTR of pages at a similar ranking position.

- Expected CTR = the median CTR of pages in the same narrow position band (positions 0–3, 3–5, 5–7, 7–10, and so on). I use narrow bands instead of the wide position_tier because CTR changes a lot inside one tier: page_1 covers positions 3–10, and typical CTR drops from about 0.32% at positions 3–5 to 0.18% at positions 7–10.
- Missed clicks = impressions × (expected CTR − actual CTR) / 100. This adjusts for both position and volume, as the lane guide recommends: a small gap on a page with huge impressions matters more than a big gap on a tiny page. A negative value means the page does better than typical.

This is a score I defined myself, not an observed outcome, so I will use it to rank pages, not as a label to train a model on.

One oddity: in this dataset, pages at positions 0–3 have a lower typical CTR (0.19%) than positions 3–5 (0.32%). The data dictionary warns that the top_3 group has low traffic in this slice, so I treat that band with caution.

In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/jjcodes999/flyrank-ml-repo/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 1. Keep pages with enough impressions and real position data
lane = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

# 2. Recalculate CTR exactly (the ctr column is rounded to 2 decimals)
lane["ctr_exact"] = lane["clicks_90d"] / lane["impressions_90d"] * 100

# 3. Put pages into narrow position groups ("bands")
bands = [0, 3, 5, 7, 10, 15, 20, 30, 50, 1000]
lane["position_band"] = pd.cut(lane["avg_position"], bands)

# 4. Expected CTR = the typical (median) CTR of pages in the same band
lane["expected_ctr"] = lane.groupby("position_band", observed=True)["ctr_exact"].transform("median")

# 5. Gap = how far below typical this page is (in percentage points)
lane["ctr_gap_pp"] = lane["expected_ctr"] - lane["ctr_exact"]

# 6. Missed clicks = how many more clicks it would have got at the typical CTR
lane["missed_clicks"] = lane["impressions_90d"] * lane["ctr_gap_pp"] / 100

print("Typical CTR in each position band:")
display(lane.groupby("position_band", observed=True)
        .agg(pages=("ctr_exact", "size"), expected_ctr=("expected_ctr", "first"))
        .round(3))

print("Top 10 pages by missed clicks:")
display(lane.sort_values("missed_clicks", ascending=False)
        [["content_id", "avg_position", "impressions_90d", "clicks_90d",
          "ctr_exact", "expected_ctr", "missed_clicks"]]
        .head(10).round(2))

Typical CTR in each position band:


,pages,expected_ctr
position_band,,
"(0, 3]",555,0.190
"(3, 5]",1964,0.318
"(5, 7]",3009,0.235
"(7, 10]",3687,0.180
"(10, 15]",3524,0.161
"(15, 20]",2352,0.141
"(20, 30]",3333,0.101
"(30, 50]",2704,0.022
"(50, 1000]",878,0.000


Top 10 pages by missed clicks:


,content_id,avg_position,impressions_90d,clicks_90d,ctr_exact,expected_ctr,missed_clicks
6653,content_5fe46e04994d,4.2,517715,741,0.14,0.32,907.86
22028,content_73c54f78c06a,4.7,213963,211,0.10,0.32,470.45
7678,content_8451fc6f034d,2.3,272144,75,0.03,0.19,443.00
29879,content_1a9e894be2e2,4.0,416180,944,0.23,0.32,381.48
3394,content_36ff89c8214e,7.3,295097,154,0.05,0.18,378.59
7445,content_c8e9d6ab9013,9.7,208678,0,0.00,0.18,376.62
6903,content_c84a0ab98e90,7.8,223271,70,0.03,0.18,332.96
15914,content_0919dd345d80,7.0,119217,26,0.02,0.23,254.06
26474,content_a7427266c305,5.7,201111,219,0.11,0.23,253.45
5621,content_97a86caf3a3d,6.4,147670,97,0.07,0.23,249.91


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Answer:

My success metric is Precision@20: of the top 20 pages my score recommends, what fraction were still getting fewer clicks than typical for their position in a later time period? I chose top 20 because an editor can realistically review about 20 pages at a time.

The data has no column saying a page was truly worth reviewing, but I can still check my ranking against something observed later:
1. I rank pages using only the first 60 days (90-day totals minus the last 30 days).
2. I then check the last 30 days: was each page still below the typical CTR for its position band? That is the "hit" for Precision@20.

"Good" means clearly beating two comparisons: a random pick (the base rate) and the repo's simple fixed rule (impressions ≥ 500, position ≤ 20, CTR < 0.5%).

Results (code below): random pick 0.42, fixed rule 0.80, missed-clicks score 1.00 at Precision@20. At Precision@200, the fixed rule drops to 0.70 while the missed-clicks score stays at 0.93.

Honest limits:
- This is an easy test for high-traffic pages, because their CTR is naturally stable from month to month. So a high score shows the flags are real and lasting, not random noise. It does not prove that editing these pages would bring more clicks.
- avg_position covers the full 90 days, so it slightly overlaps the checking period. With the warehouse data later, I will keep the two time periods fully separate.

In [9]:
# Check the ranking against what happened LATER

check = lane.copy()

# Split time: "early" = first 60 days, "late" = last 30 days
check["impressions_early"] = check["impressions_90d"] - check["impressions_last_30d"]
check["clicks_early"] = check["clicks_90d"] - check["clicks_last_30d"]

# Keep pages with enough traffic in BOTH periods
check = check[(check["impressions_early"] >= 100) & (check["impressions_last_30d"] >= 50)].copy()

check["ctr_early"] = check["clicks_early"] / check["impressions_early"] * 100
check["ctr_late"] = check["clicks_last_30d"] / check["impressions_last_30d"] * 100

by_band = check.groupby("position_band", observed=True)

# FORECAST: missed clicks, using only the early period
check["score_early"] = check["impressions_early"] * (by_band["ctr_early"].transform("median") - check["ctr_early"]) / 100

# CHECK: was the page still below typical CTR in the late period? (1 = yes, 0 = no)
check["still_under"] = (check["ctr_late"] < by_band["ctr_late"].transform("median")).astype(int)

# COMPARISON: the repo's simple fixed rule, ranked by impressions
is_flagged = (check["impressions_90d"] >= 500) & (check["avg_position"] <= 20) & (check["ctr"] < 0.5)
check["rule_score"] = is_flagged * check["impressions_90d"]

def precision_at_k(score_column, k):
    top_k = check.sort_values(score_column, ascending=False).head(k)
    return top_k["still_under"].mean()

print("Pages checked:", len(check))
print("Random pick (base rate):     ", round(check["still_under"].mean(), 2))
print("Fixed rule      Precision@20:", round(precision_at_k("rule_score", 20), 2))
print("Missed clicks   Precision@20:", round(precision_at_k("score_early", 20), 2))
print("Fixed rule      Precision@200:", round(precision_at_k("rule_score", 200), 2))
print("Missed clicks   Precision@200:", round(precision_at_k("score_early", 200), 2))

Pages checked: 18162
Random pick (base rate):      0.42
Fixed rule      Precision@20: 0.8
Missed clicks   Precision@20: 1.0
Fixed rule      Precision@200: 0.7
Missed clicks   Precision@200: 0.93


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Answer:

My unit of analysis is one content page. Each row represents one page and contains information about its search visibility, clicks, CTR, ranking position, content properties and engagement.

For this lane, I am currently keeping pages with at least 100 impressions and valid ranking-position data so that very low-volume pages do not dominate the analysis. content_id identifies the page but would not be used as an ML feature.

client_id is also an identifier only. It will be used for grouping and for testing on clients the model has never seen, but never as a feature. This matters because all comparison articles come from a single client.

In [ ]:
columns_to_show = [
    "content_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr_exact",
    "avg_position",
    "position_band",
    "content_age_days",
    "days_since_last_update",
    "expected_ctr",
    "missed_clicks"
]

display(lane[columns_to_show].head(10).round(2))

print("Rows:", len(lane))
print("Unique pages:", lane["content_id"].nunique())
print("One row = one content page")

,content_id,content_type,impressions_90d,clicks_90d,ctr_exact,avg_position,position_band,content_age_days,days_since_last_update,expected_ctr,missed_clicks
0,content_304f48230142,keyword article,3803,29,0.76,10.6,"(10, 15]",187,20,0.16,-22.88
1,content_a1fb4e703a9e,keyword article,15320,7,0.05,20.3,"(20, 30]",445,25,0.10,8.44
2,content_9aa793d4d895,keyword article,12581,11,0.09,36.5,"(30, 50]",141,20,0.02,-8.22
3,content_331d6c4de07b,keyword article,11751,58,0.49,6.2,"(5, 7]",463,22,0.23,-30.39
4,content_d99b7a2d90ca,keyword article,19140,24,0.13,44.0,"(30, 50]",263,14,0.02,-19.77
5,content_d4084a4bc775,keyword article,3970,1,0.03,8.5,"(7, 10]",147,20,0.18,6.17
7,content_a63219c6e95a,keyword article,1724,1,0.06,21.2,"(20, 30]",445,22,0.10,0.74
8,content_5e6c160719bc,keyword article,32574,29,0.09,46.0,"(30, 50]",90,20,0.02,-21.80
9,content_c27558df2b0c,keyword article,1240,2,0.16,4.9,"(3, 5]",257,104,0.32,1.95
10,content_d8ee6cc6d642,keyword article,20919,324,1.55,2.2,"(0, 3]",329,104,0.19,-284.18


Rows: 22006
Unique pages: 22006
One row = one content page


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Answer:

A fixed rule could flag pages using thresholds such as low CTR, high impressions and a strong ranking position. This would be a useful baseline, but it may be too simple because CTR opportunity can depend on several signals at the same time, including ranking position, impressions, content type, intent, age, freshness and engagement.

A ranking or scoring method could combine these signals instead of relying on one set of hard cutoffs. However, I would only say that ML is better if it performs better than the fixed-rule baseline on data it was not trained on. If it does not, the simpler rule would be preferable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.